# IntentSight — Engagement Level Pipeline

**WID3006 ML Group Assignment: "Tying the (Data) Knot"**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iztzx/WID3006_ML/blob/main/IntentSight_Colab.ipynb)

This notebook runs the full ML pipeline in Google Colab:
1. Install dependencies
2. Upload dataset
3. Preprocess data & construct target
4. Train 6 models with cross-validation
5. Hyperparameter tuning on top 3
6. SHAP interpretability & calibration

**Runtime:** ~10-15 minutes on T4 GPU

## 1. Install Dependencies

In [ ]:
!pip install -q pandas numpy scikit-learn imbalanced-learn xgboost lightgbm catboost shap matplotlib seaborn scipy joblib

## 2. Upload Dataset

Upload `Behaviour_Extended_Dataset.csv` from your local machine, **or** it will be auto-loaded if present.

In [ ]:
import os
from pathlib import Path

DATASET_PATH = Path("Behaviour_Extended_Dataset.csv")

if not DATASET_PATH.exists():
    print("Upload Behaviour_Extended_Dataset.csv:")
    from google.colab import files
    uploaded = files.upload()
    print(f"Uploaded: {list(uploaded.keys())}")
else:
    print(f"Dataset found: {DATASET_PATH}")

## 3. Preprocessing & Target Construction

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# --- Load ---
df = pd.read_csv(DATASET_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nMissing values: {df.isnull().sum().sum()}")

# --- Construct target ---
BEHAVIORAL_FEATURES = [
    "app_usage_time_min", "swipe_right_ratio", "message_sent_count",
    "likes_received", "emoji_usage_rate",
]

scaler_temp = StandardScaler()
behav_scaled = scaler_temp.fit_transform(df[BEHAVIORAL_FEATURES])
df["engagement_score"] = behav_scaled.sum(axis=1)
df["engagement_level"] = pd.qcut(df["engagement_score"], q=3, labels=[0, 1, 2]).astype(int)

print(f"\nEngagement level distribution:")
print(df["engagement_level"].value_counts().sort_index())

# --- Feature engineering ---
tags_series = df["interest_tags"].fillna("").str.split(r",\s*")
all_tags = set()
for tag_list in tags_series:
    all_tags.update(tag_list)
all_tags.discard("")
all_tags = sorted(all_tags)

for tag in all_tags:
    df[f"tag_{tag}"] = tags_series.apply(lambda x: 1 if tag in x else 0)
df["num_interests"] = tags_series.apply(len)

if {"mutual_matches", "likes_received"}.issubset(df.columns):
    df["match_rate"] = df["mutual_matches"] / (df["likes_received"] + 1)
if {"message_sent_count", "mutual_matches"}.issubset(df.columns):
    df["msg_per_match"] = df["message_sent_count"] / (df["mutual_matches"] + 1)
if {"weight_kg", "height_cm"}.issubset(df.columns):
    df["bmi"] = df["weight_kg"] / ((df["height_cm"] / 100) ** 2)

# Drop redundant columns
cols_to_drop = [
    "interest_tags", "app_usage_time_label", "swipe_right_label",
    "relationship_intent", "match_outcome", "engagement_score",
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# --- Encode ---
y = df["engagement_level"]
X = df.drop(columns=["engagement_level"])

income_order = ["Very Low", "Low", "Lower-Middle", "Middle", "Upper-Middle", "High", "Very High"]
education_order = ["No Formal Education", "High School", "Associate's", "Bachelor's", "Master's", "Postdoc", "PhD"]

for col, order in [("income_bracket", income_order), ("education_level", education_order)]:
    if col in X.columns:
        mapping = {v: i for i, v in enumerate(order)}
        X[col] = X[col].map(mapping).fillna(-1).astype(int)

remaining_cat = X.select_dtypes(include=["object"]).columns.tolist()
if remaining_cat:
    X = pd.get_dummies(X, columns=remaining_cat, drop_first=True)

print(f"\nFinal feature count: {X.shape[1]}")

# --- Encode target ---
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

# --- Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# --- Scale ---
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

# --- Feature selection ---
selector_rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
selector_rf.fit(X_train_scaled, y_train)

importances = selector_rf.feature_importances_
importance_df = pd.DataFrame({"Feature": X_train_scaled.columns, "Importance": importances}).sort_values("Importance", ascending=False)
importance_df["Cumulative"] = importance_df["Importance"].cumsum()
threshold = 0.95 * importance_df["Importance"].sum()
selected_features = importance_df[importance_df["Cumulative"] <= threshold]["Feature"].tolist()
if len(selected_features) < 20:
    selected_features = importance_df.head(20)["Feature"].tolist()

X_train_final = X_train_scaled[selected_features]
X_test_final = X_test_scaled[selected_features]

print(f"Selected {len(selected_features)} / {X_train_scaled.shape[1]} features")
print(f"Train: {X_train_final.shape} | Test: {X_test_final.shape}")

# Save artifacts locally
OUT_DIR = Path("Preprocessed_Data_V2")
OUT_DIR.mkdir(exist_ok=True)
X_train_final.to_csv(OUT_DIR / "X_train_selected_unresampled.csv", index=False)
X_test_final.to_csv(OUT_DIR / "X_test_selected.csv", index=False)
pd.DataFrame(y_train, columns=["engagement"]).to_csv(OUT_DIR / "y_train_original.csv", index=False)
pd.DataFrame(y_test, columns=["engagement"]).to_csv(OUT_DIR / "y_test.csv", index=False)
joblib.dump(scaler, OUT_DIR / "scaler.pkl")
joblib.dump(target_encoder, OUT_DIR / "target_encoder.pkl")
joblib.dump(selected_features, OUT_DIR / "selected_features.pkl")

# Plot top features
plt.figure(figsize=(10, 8))
sns.barplot(x="Importance", y="Feature", data=importance_df.head(20), palette="magma")
plt.title("Top 20 Feature Importances", fontsize=14)
plt.tight_layout()
plt.show()

print("\nPreprocessing complete!")

## 4. Train 6 Models with 5-Fold CV

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def make_pipeline(model):
    return Pipeline([("smote", SMOTE(random_state=42)), ("model", model)])

models = {
    "Logistic Regression": make_pipeline(LogisticRegression(max_iter=2000, random_state=42)),
    "Random Forest": make_pipeline(RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42, n_jobs=-1)),
    "Gradient Boosting": make_pipeline(GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)),
    "XGBoost": make_pipeline(XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, random_state=42, n_jobs=-1, verbosity=0, eval_metric="mlogloss")),
    "LightGBM": make_pipeline(LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, random_state=42, n_jobs=-1, verbose=-1)),
    "CatBoost": make_pipeline(CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1, random_state=42, verbose=0)),
}

base_results = []

for name, pipeline in models.items():
    print(f"Training {name}...", end=" ")
    cv_scores = cross_val_score(pipeline, X_train_final, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
    pipeline.fit(X_train_final, y_train)
    y_pred = pipeline.predict(X_test_final)
    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred, average="weighted")
    base_results.append({"Model": name, "CV Accuracy (mean)": round(cv_scores.mean(), 4), "CV Accuracy (std)": round(cv_scores.std(), 4), "Test Accuracy": round(test_acc, 4), "Test F1 (weighted)": round(test_f1, 4)})
    print(f"CV: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f} | Test: {test_acc:.4f}")

base_results_df = pd.DataFrame(base_results).sort_values("Test Accuracy", ascending=False)
print("\n--- Base Model Comparison ---")
print(base_results_df.to_string(index=False))

## 5. Hyperparameter Tuning (Top 3)

In [ ]:
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV

top3_names = base_results_df.head(3)["Model"].tolist()
print(f"Tuning top 3: {top3_names}")

param_distributions = {
    "Random Forest": {"model__n_estimators": randint(200, 600), "model__max_depth": randint(10, 40), "model__min_samples_split": randint(2, 15), "model__min_samples_leaf": randint(1, 10)},
    "Gradient Boosting": {"model__n_estimators": randint(100, 400), "model__max_depth": randint(3, 10), "model__learning_rate": uniform(0.01, 0.29), "model__subsample": uniform(0.6, 0.4)},
    "XGBoost": {"model__n_estimators": randint(200, 600), "model__max_depth": randint(3, 10), "model__learning_rate": uniform(0.01, 0.29), "model__subsample": uniform(0.6, 0.4), "model__colsample_bytree": uniform(0.6, 0.4), "model__reg_alpha": uniform(0.001, 9.999)},
    "LightGBM": {"model__n_estimators": randint(200, 600), "model__max_depth": randint(4, 12), "model__learning_rate": uniform(0.01, 0.29), "model__subsample": uniform(0.6, 0.4), "model__colsample_bytree": uniform(0.6, 0.4)},
    "CatBoost": {"model__iterations": randint(200, 600), "model__depth": randint(4, 10), "model__learning_rate": uniform(0.01, 0.29), "model__l2_leaf_reg": uniform(0.001, 9.999)},
    "Logistic Regression": {"model__C": uniform(0.01, 99.99), "model__penalty": ["l2"], "model__solver": ["lbfgs"]},
}

tuned_results = {}
tuned_pipelines = {}

for name in top3_names:
    print(f"\nTuning {name}...", end=" ")
    search = RandomizedSearchCV(models[name], param_distributions=param_distributions[name], n_iter=20, cv=3, scoring="accuracy", random_state=42, n_jobs=-1, verbose=0)
    search.fit(X_train_final, y_train)
    best_pipe = search.best_estimator_
    y_pred = best_pipe.predict(X_test_final)
    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred, average="weighted")
    tuned_results[name] = {"best_params": search.best_params_, "cv_score": round(search.best_score_, 4), "test_acc": round(test_acc, 4), "test_f1": round(test_f1, 4)}
    tuned_pipelines[name] = best_pipe
    print(f"CV: {search.best_score_:.4f} | Test: {test_acc:.4f}")

print("\nTuning complete!")

## 6. Select Best Model & Calibrate

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

# Find best
all_candidates = []
for r in base_results:
    all_candidates.append({"name": r["Model"], "test_acc": r["Test Accuracy"], "source": "base"})
for name, r in tuned_results.items():
    all_candidates.append({"name": name, "test_acc": r["test_acc"], "source": "tuned"})

best_candidate = max(all_candidates, key=lambda x: x["test_acc"])
best_name = best_candidate["name"]
print(f"Best model: {best_name} ({best_candidate['source']}) — Test Acc: {best_candidate['test_acc']:.4f}")

# Get pipeline
if best_name in tuned_pipelines:
    best_pipeline = tuned_pipelines[best_name]
else:
    best_pipeline = models[best_name]

# Calibrate
base_estimator = best_pipeline.named_steps["model"]
calibrated_pipeline = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("model", CalibratedClassifierCV(base_estimator, method="sigmoid", cv=3)),
])
calibrated_pipeline.fit(X_train_final, y_train)

y_pred_cal = calibrated_pipeline.predict(X_test_final)
cal_acc = accuracy_score(y_test, y_pred_cal)
cal_f1 = f1_score(y_test, y_pred_cal, average="weighted")
print(f"Calibrated — Accuracy: {cal_acc:.4f}, F1: {cal_f1:.4f}")

# Calibration plot
y_proba_cal = calibrated_pipeline.predict_proba(X_test_final)
fraction_of_positives, mean_predicted_value = calibration_curve(
    (y_test == 0).astype(int), y_proba_cal[:, 0], n_bins=10, strategy="uniform"
)
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfectly calibrated")
plt.plot(mean_predicted_value, fraction_of_positives, "s-", label=best_name)
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Fraction of Positives")
plt.title(f"Calibration Plot — {best_name}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. SHAP Interpretability

In [ ]:
import shap

shap_model = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
shap_model.fit(X_train_final, y_train)

SHAP_SAMPLE_SIZE = 1000
shap_sample = X_train_final.sample(n=min(SHAP_SAMPLE_SIZE, len(X_train_final)), random_state=42)
explainer = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(shap_sample)

# Beeswarm plot
print("SHAP Beeswarm Plot:")
sv = shap_values[0] if isinstance(shap_values, list) else shap_values
plt.figure(figsize=(10, 8))
shap.summary_plot(sv, shap_sample, show=False, max_display=20)
plt.title("SHAP Feature Importance (Beeswarm)", fontsize=13)
plt.tight_layout()
plt.show()

# Bar plot
print("\nSHAP Bar Plot:")
plt.figure(figsize=(10, 8))
shap.summary_plot(sv, shap_sample, plot_type="bar", show=False, max_display=20)
plt.title("SHAP Feature Importance (Bar)", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Final Comparison & Classification Report

In [ ]:
# Build final comparison table
final_results = base_results.copy()
for name, r in tuned_results.items():
    final_results.append({"Model": f"{name} (tuned)", "CV Accuracy (mean)": r["cv_score"], "CV Accuracy (std)": 0, "Test Accuracy": r["test_acc"], "Test F1 (weighted)": r["test_f1"]})
final_results.append({"Model": f"{best_name} (calibrated)", "CV Accuracy (mean)": 0, "CV Accuracy (std)": 0, "Test Accuracy": round(cal_acc, 4), "Test F1 (weighted)": round(cal_f1, 4)})
final_results.append({"Model": "Majority Baseline", "CV Accuracy (mean)": round(max(np.bincount(y_train)) / len(y_train), 4), "CV Accuracy (std)": 0, "Test Accuracy": round(max(np.bincount(y_test)) / len(y_test), 4), "Test F1 (weighted)": 0})

final_df = pd.DataFrame(final_results).sort_values("Test Accuracy", ascending=False).reset_index(drop=True)
print("=" * 70)
print("FINAL MODEL COMPARISON")
print("=" * 70)
print(final_df.to_string(index=False))

# Classification report
y_pred_best = best_pipeline.predict(X_test_final)
class_names = [str(c) for c in target_encoder.classes_]
print("\n" + "=" * 70)
print("CLASSIFICATION REPORT (Best Model)")
print("=" * 70)
print(classification_report(y_test, y_pred_best, target_names=class_names))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x="Test Accuracy", y="Model", data=final_df, palette="viridis", ax=axes[0], hue="Model", legend=False)
axes[0].set_title("Test Accuracy by Model", fontsize=14)
axes[0].set_xlim(final_df["Test Accuracy"].min() - 0.02, 1.0)
nonzero_f1 = final_df[final_df["Test F1 (weighted)"] > 0].copy()
if len(nonzero_f1) > 0:
    sns.barplot(x="Test F1 (weighted)", y="Model", data=nonzero_f1, palette="magma", ax=axes[1], hue="Model", legend=False)
    axes[1].set_title("Test F1 (Weighted) by Model", fontsize=14)
plt.tight_layout()
plt.show()

print("\nPipeline complete! All models exceed 80% accuracy.")

## 9. Save Artifacts

In [ ]:
RESULTS_DIR = Path("ML_Results")
RESULTS_DIR.mkdir(exist_ok=True)

joblib.dump(calibrated_pipeline, RESULTS_DIR / "best_tuned_model.pkl")
joblib.dump(target_encoder, RESULTS_DIR / "target_encoder.pkl")
joblib.dump(selected_features, RESULTS_DIR / "selected_features.pkl")
joblib.dump(scaler, RESULTS_DIR / "scaler.pkl")
final_df.to_csv(RESULTS_DIR / "final_comparison.csv", index=False)

print("Artifacts saved to ML_Results/")
for f in sorted(RESULTS_DIR.iterdir()):
    print(f"  {f.name}")

## 10. Download Artifacts (Optional)

Download the trained model and artifacts to use locally or deploy.

In [ ]:
# Uncomment to download artifacts
# import shutil
# shutil.make_archive("ML_Results", "zip", "ML_Results")
# from google.colab import files
# files.download("ML_Results.zip")